In [ ]:
import operator
#dic of values for button positions
#trying to get the following behaviour
#first button selects the top level value to be changed
#second button usually selects the rough values for the value to be changed (but can also select variable and value!)
#third button controlls modifiers to the rough values, which can be
#adding a smaller value to or multiplying the rough value by a certain amount.

#naming convention: dABc is an array returning values based on some button position
#A = logical position of the first button
#B = 0 - rough value 1 - fine value
#C = used only when the position of the corresponding dAB also selects form multiple variables
#in principle, each array has 4 elements with indices [0,1,2,3] which are selected by the position of a button
#taking the logic values [0,1,2,3]
#example:
#d10 - an arrau of 4 values which are returned
#    - when the first button is on logical position 1 (A=1), 
#    - and represent rough values (B=0) (which are controlled by the second button)

#d11 - an array of 4 values which are returned
#    - when the first button is on logical position 1 (A=1),
#    - represent a modifier (B=1) which in our case are smaller values which will be added to d10
#    - controlled by the third button
#d20 is a special case, as the first two elements of this array correpond to integration time and the last two
#correspond to frequency.
#d210 affects only the d20[0,1] while d211 works with d20[2,3], where the brackets denote indices not values
#the modifier (adding fine values or multiplication) is controlled from var_dic
#in the case of d10 for example, the addition operator (operator.add) is paired to every element of d00,
#meaning in the code the positions of the second and third button returns the sum of corresponding elements
#d00[i] + d01[j]
#in the case of d20, a value from d20[0,1] is multiplied with a value from d210
#while a value from d20[2,3] is added to a value from d211 


#the strings like ['hr_angle'] or ['obj_dec'] help identify which variables is assigned the selected values
#and to understand what each combiunation dABC elements is assigned to
#the [hr] [deg] strings are for display
#the d30 d31 arrays are used for filename selection. Addition of strings concatenates.

#the key values of var dic correspond to positions in the first button.



d00 = [-6,-3,0,+3] #rough values for HA
d01 = [0,1,2,3] #modifiers for HA
d10 = [-80,-50,-30,-10] #rough values for dec
d11 = [0,5,15,20] #modifiers for dec
d20 = [0.2,1,50,500] #rough values for: first two values - int time, last two - frequency
d210 = [1,2,3,12] #modifier for int time
d211 = [0,50,100,250] #modifier for freq
d30 = ['Galaxy','Planet','BH','Misc'] #rough value for file name (first part of filename)
d31 = ['1.jpg','2.jpg','3.jpg','4.jpg'] #modifier for filename (last part of filename)
var_dic = \
	{0:[['hr_angle']*4,list(zip([operator.add]*4,d00)),[d01]*4,["[hr]"]*4], #the d00 d01 values are added
	 1:[['obj_dec']*4,list(zip([operator.add]*4,d10)),[d11]*4,["[deg]"]*4], #the d10 d11 values are added
	 2:[['int_time']*2+ ['obs_freq']*2,list(zip([operator.mul]*2+[operator.add]*2,d20)),[d210]*2+[d211]*2,["[hr]"]*2+["[GHz]"]*2], #explained above, addition or multipl
	 3:[['img_file']*4,list(zip([operator.add]*4,d30)),[d31]*4,[" "]*4]} #string values are concatenated from values in d30 and d31.
	
swi_bits = [52,44,36]



Note: you may need to restart the kernel to use updated packages.


In [ ]:
"""
Detaillierte Erklärung von la_config.py / la_config.ipynb:

Dieses Modul definiert die Steuerlogik für die physische Bedienung des Systems.
Die Idee ist, dass drei Drehknöpfe bzw. Schalter in Kombination verschiedene
Parameter der Simulation einstellen.

Die Architektur ist wie folgt:

1. Erste Knopfposition (erste Taste) wählt das zu ändernde Feld:
   - Position 0 = Stundenwinkel (HA)
   - Position 1 = Deklination (Dec)
   - Position 2 = Integrationszeit / Beobachtungsfrequenz
   - Position 3 = Bilddatei

2. Zweite Knopfposition wählt einen groben Wert aus einem Werte-Array.
3. Dritte Knopfposition wählt einen Feinstellwert aus einem zweiten Array.

Für die meisten Variablen gilt:
- Großer Wert = dA0
- Kleiner Wert = dA1
- Operator = add oder mul

Variable HA (A=0):
- d00 = [-6, -3, 0, +3]  -> grobe Stundenwinkel-Werte in Stunden
- d01 = [0, 1, 2, 3]     -> Feinwert für Stundenwinkel
- Ergebnis: d00[i] + d01[j]

Variable Dec (A=1):
- d10 = [-80, -50, -30, -10] -> grobe Deklination in Grad
- d11 = [0, 5, 15, 20]      -> Feinschritte in Grad
- Ergebnis: d10[i] + d11[j]

Variable Integrationszeit / Frequenz (A=2):
- d20 = [0.2, 1, 50, 500]
    * Index 0,1 = Integrationszeitwerte
    * Index 2,3 = Frequenzwerte
- d210 = [1, 2, 3, 12]    -> Multiplikatoren für Integrationszeit
- d211 = [0, 50, 100, 250] -> Additive Werte für Frequenz
- Ergebnis:
    * Für die ersten zwei Werte: d20[k] * d210[l]
    * Für die letzten zwei Werte: d20[k] + d211[l]

Variable Bildauswahl (A=3):
- d30 = ['Galaxy', 'Planet', 'BH', 'Misc']
- d31 = ['1.jpg', '2.jpg', '3.jpg', '4.jpg']
- Ergebnis: Zeichenketten werden aneinandergereiht,
  z.B. 'Galaxy' + '1.jpg' -> 'Galaxy1.jpg'

Das Wörterbuch `var_dic` beschreibt die Nutzung dieser Werte:
- Schlüssel 0..3 entspricht der Position des ersten Knopfs
- Wert 0: Liste mit dem Namen der Variable
- Wert 1: Liste aus Tupeln (Operator, Basiswerte)
- Wert 2: Liste mit Feinwert-Arrays
- Wert 3: Liste mit Einheiten oder Text für die Anzeige

Beispiel: `var_dic[0]` für HA
- ['hr_angle']       => Variable heißt hr_angle
- list(zip([operator.add]*4, d00)) => jeder Grobwert wird mit + verknüpft
- [d01]*4            => vier Listen mit Feinstufen
- ['[hr]']*4         => Anzeige in Stunden

`var_dic[2]` für Integrationszeit/Frequenz
- Für die ersten zwei Positionen wird mit `operator.mul` multipliziert
- Für die letzten zwei Positionen wird mit `operator.add` addiert
- So kann eine Knopfkombination entweder die Integrationszeit oder die Frequenz steuern

Schaltungsschalter:
- `swi_bits = [52, 44, 36]`
- Diese Bitpositionen werden separat als einfache Schalterzustände genutzt.

Zusammenfassung:
- la_config legt fest, welche Werte jeder Knopf-Kombination zugeordnet sind.
- Es trennt grobe Werte von feinen Modifikationen.
- Es definiert, wie Werte kombiniert werden (Addition oder Multiplikation).
- Es erzeugt damit die Basis für die Steuerung der Simulation in la_main / la_obs_img.
"""
